# SRBC Docket Lookup
Download SRBC approval PDFs for all docket numbers that appear in the junction table
and parse each for the approved source name and precise withdrawal coordinates.

Output: `data/srbc_docket_info.parquet` — one row per source per docket with
`docket_no`, `approved_source`, `lat`, `lon`, `subbasin`, `county`, `municipality`, `approval_type`.

In [ ]:
import pandas as pd
import re
import io
import time
import urllib.request
import pdfplumber

jct = pd.read_parquet('data/well_junction_table.parquet')
print('Junction rows:', len(jct))

## Extract and clean docket numbers

In [ ]:
def get_docket_clean(s):
    """Extract and normalize SRBC docket number from a planSource string.
    
    Handles OCR corruption like '201.60902' -> '20160902'
    and sub-docket suffixes like '20121202-1' -> '20121202'.
    Returns None if no valid 8-digit docket can be recovered.
    """
    if pd.isna(s):
        return None
    m = re.search(r'SRBC\s+Docket\s+(?:No\.|Number|#)?\s*([\d\.,\-]+)', str(s), re.IGNORECASE)
    if not m:
        return None
    raw = m.group(1).strip().rstrip('-.,')  # strip trailing punctuation
    cleaned = re.sub(r'[.,]', '', raw)       # remove OCR-error separators
    cleaned = re.sub(r'-\d+$', '', cleaned)  # strip sub-docket suffix (-1, -2)
    return cleaned if re.match(r'^\d{8}$', cleaned) else None


srbc_mask = jct['planSource'].str.contains(r'srbc\s+docket', case=False, na=False)
srbc_rows = jct[srbc_mask].copy()
srbc_rows['docket_no'] = srbc_rows['planSource'].apply(get_docket_clean)

dockets = sorted(srbc_rows['docket_no'].dropna().unique())
print(f'SRBC junction rows:         {len(srbc_rows):,}')
print(f'Rows with clean docket:     {srbc_rows["docket_no"].notna().sum():,}')
print(f'Rows with bad/missing docket: {srbc_rows["docket_no"].isna().sum():,}')
print(f'Unique dockets to fetch:    {len(dockets)}')

## PDF parser

Each SRBC approval PDF follows a consistent table structure:

```
Project Information
  Project Sponsor:  ...
  County:           ...
  Municipality:     ...

Source Information
  Approved Source:  ...
  Subbasin:         ...
  Withdrawal Location (degrees): Lat: XX.XXXXXX N  Long: XX.XXXXXX W
```

Some dockets cover multiple sources (multiple `Source Information` tables).

In [ ]:
# Patterns for parsing the approval document text
_PAT_COUNTY      = re.compile(r'^County:\s+(.+)$',          re.MULTILINE)
_PAT_MUNIC       = re.compile(r'^Municipality:\s+(.+)$',    re.MULTILINE)
_PAT_SOURCE      = re.compile(r'^Approved Source:\s+(.+)$', re.MULTILINE)
_PAT_SUBBASIN    = re.compile(r'^Subbasin:\s+(.+)$',        re.MULTILINE)
_PAT_LATLON      = re.compile(
    r'Withdrawal Location.*?Lat:\s*([\d.]+)\s*N\s+Long:\s*([\d.]+)\s*W',
    re.DOTALL
)
_PAT_DOCKET      = re.compile(r'Docket No\.\s+(\d[\d\-]*)')
_PAT_ATYPE       = re.compile(
    r'(Surface Water Withdrawal|Groundwater Withdrawal|'
    r'Consumptive Use|Diversion)',
    re.IGNORECASE
)


def _first(pattern, text):
    m = pattern.search(text)
    return m.group(1).strip() if m else None


def parse_srbc_pdf(pdf_bytes):
    """Parse an SRBC approval PDF and return a list of dicts (one per source).

    A single docket can contain multiple sources (multiple Source Information
    tables). We split the full text on that header and parse each section.
    Sections that contain no 'Approved Source:' line are skipped — they are
    section-heading occurrences of the phrase, not actual data tables.
    """
    full_text = ''
    with pdfplumber.open(io.BytesIO(pdf_bytes)) as pdf:
        for page in pdf.pages:
            t = page.extract_text()
            if t:
                full_text += '\n' + t

    if not full_text.strip():
        return []

    # Project-level fields shared across all sources in this docket
    docket_no = _first(_PAT_DOCKET, full_text)
    county    = _first(_PAT_COUNTY,  full_text)
    munic     = _first(_PAT_MUNIC,   full_text)
    atype_m   = _PAT_ATYPE.search(full_text)
    atype     = atype_m.group(1) if atype_m else None

    sections = re.split(r'Source Information', full_text)
    records = []
    for sec in sections[1:]:
        source_name = _first(_PAT_SOURCE, sec)
        if source_name is None:  # section heading, not a data table — skip
            continue
        ll = _PAT_LATLON.search(sec)
        records.append({
            'docket_no':       docket_no,
            'approved_source': source_name,
            'lat':             float(ll.group(1)) if ll else None,
            'lon':             -float(ll.group(2)) if ll else None,  # West → negative
            'subbasin':        _first(_PAT_SUBBASIN, sec),
            'county':          county,
            'municipality':    munic,
            'approval_type':   atype,
        })

    # Fallback: PDF with no 'Source Information' table header
    if not records:
        ll = _PAT_LATLON.search(full_text)
        records.append({
            'docket_no':       docket_no,
            'approved_source': _first(_PAT_SOURCE, full_text),
            'lat':             float(ll.group(1)) if ll else None,
            'lon':             -float(ll.group(2)) if ll else None,
            'subbasin':        _first(_PAT_SUBBASIN, full_text),
            'county':          county,
            'municipality':    munic,
            'approval_type':   atype,
        })

    return records


print('Parser defined.')

## Quick sanity-check on one known docket

In [ ]:
BASE_URL = (
    'https://www.srbc.gov/waav/Search/getdocket'
    '?projectnumber={}&documenttype=Approval&isabre=False'
)

def fetch_docket_pdf(docket_no, timeout=20):
    url = BASE_URL.format(docket_no)
    req = urllib.request.Request(url, headers={'User-Agent': 'Mozilla/5.0'})
    with urllib.request.urlopen(req, timeout=timeout) as r:
        ct = r.headers.get('Content-Type', '')
        data = r.read()
    return data if 'pdf' in ct else None


test_bytes = fetch_docket_pdf('20160902')
test_records = parse_srbc_pdf(test_bytes)
print('Test docket 20160902:')
for r in test_records:
    print(' ', r)

## Fetch and parse all 72 dockets

In [ ]:
all_records = []
fetch_errors = []
DELAY_S = 1.0  # polite crawl delay

for i, docket in enumerate(dockets):
    try:
        pdf_bytes = fetch_docket_pdf(docket)
        if pdf_bytes is None:
            fetch_errors.append((docket, 'no_pdf'))
            print(f'  [{i+1:02d}/{len(dockets)}] {docket} — no PDF returned')
        else:
            recs = parse_srbc_pdf(pdf_bytes)
            for r in recs:
                r['query_docket'] = docket  # what we queried with
            all_records.extend(recs)
            lat_info = f"lat={recs[0]['lat']:.4f}, lon={recs[0]['lon']:.4f}" if recs and recs[0]['lat'] else 'no coords'
            src_info = recs[0]['approved_source'] if recs else '?'
            print(f'  [{i+1:02d}/{len(dockets)}] {docket} — {len(recs)} source(s): "{src_info}" {lat_info}')
    except Exception as e:
        fetch_errors.append((docket, str(e)))
        print(f'  [{i+1:02d}/{len(dockets)}] {docket} — ERROR: {e}')
    time.sleep(DELAY_S)

print(f'\nDone. {len(all_records)} source records from {len(dockets)} dockets.')
print(f'Errors: {len(fetch_errors)}')

## Review and save results

In [ ]:
srbc_info = pd.DataFrame(all_records)
print('Shape:', srbc_info.shape)
print('\nCoordinate coverage:')
has_coords = srbc_info['lat'].notna()
print(f'  With lat/lon:    {has_coords.sum()} / {len(srbc_info)}')
print(f'  Without lat/lon: {(~has_coords).sum()}')
print('\nApproval types:')
print(srbc_info['approval_type'].value_counts().to_string())
print()
print(srbc_info[['query_docket','approved_source','lat','lon','county','subbasin']]
      .to_string())

In [ ]:
# Sources without coordinates — these are typically groundwater wells
no_coords = srbc_info[srbc_info['lat'].isna()]
print(f'Sources without coordinates ({len(no_coords)} rows):')
print(no_coords[['query_docket','approved_source','approval_type','county']].to_string())

In [ ]:
srbc_info.to_parquet('data/srbc_docket_info.parquet', index=False)
print('Saved: data/srbc_docket_info.parquet')
print(srbc_info.dtypes)

## Join SRBC coordinates back to junction table

For SRBC-tagged junction rows, the docket lookup gives us:
- A precise withdrawal point coordinate (better than the well proxy)
- The confirmed source name from SRBC

When a docket has multiple sources, we need to pick the right one.
The planSource feature name is the best discriminator.

In [ ]:
# Attach docket_no to all srbc junction rows
srbc_rows2 = srbc_rows.copy()
srbc_rows2['docket_no'] = srbc_rows2['planSource'].apply(get_docket_clean)

# For dockets with a single source, join is unambiguous
docket_src_count = srbc_info.groupby('query_docket').size().rename('n_sources')
single_source_dockets = docket_src_count[docket_src_count == 1].index
multi_source_dockets  = docket_src_count[docket_src_count > 1].index

print(f'Dockets with 1 source:  {len(single_source_dockets)}')
print(f'Dockets with 2+ sources: {len(multi_source_dockets)}')
if len(multi_source_dockets):
    print('Multi-source dockets:')
    for d in multi_source_dockets:
        srcs = srbc_info[srbc_info['query_docket'] == d]['approved_source'].tolist()
        print(f'  {d}: {srcs}')

In [ ]:
# Build lookup: docket_no → (lat, lon, approved_source)
# For single-source dockets, straightforward.
# For multi-source dockets, use the source whose name best matches the planSource string.

from rapidfuzz import fuzz

def best_source_for_row(row, docket_sources):
    """Pick the SRBC source whose name best matches the planSource string."""
    plan = str(row['planSource']).lower()
    best_score, best_src = -1, docket_sources.iloc[0]
    for _, src_row in docket_sources.iterrows():
        name = str(src_row['approved_source'] or '').lower()
        score = fuzz.partial_ratio(name, plan)
        if score > best_score:
            best_score, best_src = score, src_row
    return best_src


coord_records = []
for _, jrow in srbc_rows2.iterrows():
    dno = jrow['docket_no']
    if pd.isna(dno):
        coord_records.append({'planSource': jrow['planSource'],
                              'srbc_lat': None, 'srbc_lon': None,
                              'srbc_source_name': None, 'srbc_docket_no': None})
        continue

    docket_sources = srbc_info[srbc_info['query_docket'] == dno]
    if docket_sources.empty:
        coord_records.append({'planSource': jrow['planSource'],
                              'srbc_lat': None, 'srbc_lon': None,
                              'srbc_source_name': None, 'srbc_docket_no': dno})
        continue

    if len(docket_sources) == 1:
        src = docket_sources.iloc[0]
    else:
        src = best_source_for_row(jrow, docket_sources)

    coord_records.append({
        'planSource':        jrow['planSource'],
        'srbc_lat':          src['lat'],
        'srbc_lon':          src['lon'],
        'srbc_source_name':  src['approved_source'],
        'srbc_docket_no':    dno,
    })

srbc_coords = pd.DataFrame(coord_records).drop_duplicates('planSource')

print('SRBC coord lookup:')
print(f'  Unique planSources: {len(srbc_coords)}')
print(f'  With SRBC lat/lon:  {srbc_coords["srbc_lat"].notna().sum()}')
print(f'  Without coords:     {srbc_coords["srbc_lat"].isna().sum()}')

In [ ]:
srbc_coords.to_parquet('data/srbc_coords_lookup.parquet', index=False)
print('Saved: data/srbc_coords_lookup.parquet')
print(srbc_coords[['planSource','srbc_docket_no','srbc_source_name','srbc_lat','srbc_lon']]
      .sort_values('srbc_lat').to_string())